In [ ]:
!pip install torchvision


In [1]:
import os
import shutil

def organize_images_by_region_type(image_paths, output_dir):
    # Assemblies assigned to valid and test sets

    val_assemblies = {
    'Kota Lama',
    'Komtar',
    'Elopura',
    'Seri Setia',
    'Kota Darul Aman',
    'Pujut',
    'Sungai Balang',
    'Bukit Pinang',
    'Pulau Betong',
    'Chuping',
    'Sabak'
    }
    
    test_assemblies = {
        'Kota Iskandar',
        'Bukit Bintang',
        'Sungai Sibuga',
        'Bakar Arang',
        'Satok',
        'Tokai',
        'Jelai',
        'Teluk Intan',
        'Permaisuri'
    }

    water_image_counter = {'train': 0, 'valid': 0, 'test': 0}
    water_limits = {'train': 206, 'valid': 44, 'test': 45}

    def extract_region_info(folder_path, is_water=False):
        parts = os.path.normpath(folder_path).split(os.sep)
        try:
            if is_water:
                # For water images: region_type = -3, assembly = "Water"
                region_type = parts[-3]
                state = parts[-4]
                district = parts[-3]  # Assume district = same as region_type
                assembly = "Water"
            else:
                region_type = parts[-5]
                state = parts[-4]
                district = parts[-3]
                assembly = parts[-2]
            return region_type, state, district, assembly
        except IndexError:
            return None, None, None, None

    for img_path in image_paths:
        if os.sep + "images" + os.sep not in img_path and not img_path.endswith(os.sep + "images"):
            continue  # Only process images in 'images' folders

        folder_path = os.path.dirname(img_path)
        is_water = "Water" in folder_path
        region_type, state, district, assembly = extract_region_info(folder_path, is_water=is_water)

        if None in (region_type, state, district, assembly):
            print(f"Skipped: Incomplete path structure -> {folder_path}")
            continue

        #print(f"Processing: {img_path}")
        #print(f"→ Region: {region_type}, State: {state}, District: {district}, Assembly: {assembly}")

        if region_type not in ['Rural', 'Urban'] and not is_water:
            print("  Skipped: Region type not 'Rural', 'Urban', or valid Water case")
            continue

        # Determine split
        if is_water:
            assigned = False
            for split in ['train', 'valid', 'test']:
                if water_image_counter[split] < water_limits[split]:
                    dest_split = split
                    water_image_counter[split] += 1
                    assigned = True
                    print(f"  Assigned to '{split}' under Water quota. Remaining: {water_limits[split] - water_image_counter[split]}")
                    break
            if not assigned:
                print("  Skipped: Water quota full for all splits")
                continue
        else:
            if assembly in valid_assemblies:
                dest_split = 'valid'
            elif assembly in test_assemblies:
                dest_split = 'test'
            else:
                dest_split = 'train'
            #print(f"  Assigned to split: {dest_split}")

        # Final destination
        dest_folder = os.path.join(output_dir, dest_split, region_type)
        os.makedirs(dest_folder, exist_ok=True)

        dest_path = os.path.join(dest_folder, os.path.basename(img_path))
        shutil.copy2(img_path, dest_path)
        #print(f"  Copied to: {dest_path}\n")

    #print("✅ Image organization completed.")

from glob import glob
# Recursively get all image files ending with .png or .jpg (adjust as needed)
image_paths = glob(r'C:/Users/Yap Jack/OneDrive/Desktop/FYP/Datasets/FYP2_DeepEarthMY/**/*.png', recursive=True)

organize_images_by_region_type(
    image_paths=image_paths,
    output_dir='organized_dataset'
)



In [2]:
import os
from PIL import Image
from torchvision import transforms
from torchvision.utils import save_image
import torch

# Set paths
input_dir = 'organized_dataset'
output_dir = 'new_split_dataset'

# Define image preprocessing pipeline
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # Common ImageNet mean
                         std=[0.229, 0.224, 0.225])   # Common ImageNet std
])

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

# Walk through input directory
for root, _, files in os.walk(input_dir):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            input_path = os.path.join(root, file)
            relative_path = os.path.relpath(input_path, input_dir)
            output_path = os.path.join(output_dir, relative_path)
            os.makedirs(os.path.dirname(output_path), exist_ok=True)

            # Load and transform the image
            img = Image.open(input_path).convert('RGB')
            img_tensor = transform(img)

            # Save the preprocessed tensor as an image
            save_image(img_tensor, output_path)

print("Preprocessing complete. Images saved to:", output_dir)


ModuleNotFoundError: No module named 'torchvision'

In [5]:
import os
from PIL import Image
import numpy as np

# Set paths
input_dir = 'organized_dataset'
output_dir = 'preprocessed_dataset2'
img_size = (224, 224)

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Walk through all subfolders and images
for subdir, dirs, files in os.walk(input_dir):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
            img_path = os.path.join(subdir, file)
            rel_path = os.path.relpath(img_path, input_dir)
            save_path = os.path.join(output_dir, rel_path)

            os.makedirs(os.path.dirname(save_path), exist_ok=True)

            # Open, resize, and normalize the image
            img = Image.open(img_path).convert('RGB')
            img = img.resize(img_size)

            img_array = np.asarray(img).astype(np.float32) / 255.0  # Normalize to [0, 1]
            img_normalized = Image.fromarray((img_array * 255).astype(np.uint8))  # Convert back for saving

            img_normalized.save(save_path)


# DeepEarthMY V1 Split

In [2]:
import os
import shutil
import random

def organize_images_by_region_type(image_paths, output_dir, seed=42):
    random.seed(seed)  # For reproducibility
    water_images = []
    land_images = []

    def extract_region_info(folder_path, is_water=False):
        parts = os.path.normpath(folder_path).split(os.sep)
        try:
            if is_water:
                region_type = parts[-3]
                state = parts[-4]
                district = parts[-3]
                assembly = "Water"
            else:
                region_type = parts[-5]
                state = parts[-4]
                district = parts[-3]
                assembly = parts[-2]
            return region_type, state, district, assembly
        except IndexError:
            return None, None, None, None

    for img_path in image_paths:
        if os.sep + "images" + os.sep not in img_path and not img_path.endswith(os.sep + "images"):
            continue

        folder_path = os.path.dirname(img_path)
        is_water = "Water" in folder_path
        region_type, state, district, assembly = extract_region_info(folder_path, is_water=is_water)

        if None in (region_type, state, district, assembly):
            print(f"Skipped: Incomplete path structure -> {folder_path}")
            continue

        if region_type not in ['Rural', 'Urban'] and not is_water:
            print("Skipped: Region type not 'Rural', 'Urban', or valid Water case")
            continue

        if is_water:
            water_images.append((img_path, region_type))
        else:
            land_images.append((img_path, region_type))

    def split_and_copy(images, label):
        total = len(images)
        n_train = int(total * 0.8)
        n_valid = int(total * 0.1)
        n_test = total - n_train - n_valid  # Ensure full use

        random.shuffle(images)

        splits = {
            'train': images[:n_train],
            'valid': images[n_train:n_train + n_valid],
            'test': images[n_train + n_valid:]
        }

        for split, split_images in splits.items():
            for img_path, region_type in split_images:
                dest_folder = os.path.join(output_dir, split, region_type)
                os.makedirs(dest_folder, exist_ok=True)
                dest_path = os.path.join(dest_folder, os.path.basename(img_path))
                shutil.copy2(img_path, dest_path)

            print(f"✅ {label.capitalize()} images assigned to {split}: {len(split_images)}")

    split_and_copy(land_images, 'land')
    split_and_copy(water_images, 'water')

# Example usage:
from glob import glob
image_paths = glob(r'C:/Users/YapJack/Desktop/FYP/DeepEarthMY2 (Classification)/**/*.png', recursive=True)

organize_images_by_region_type(
    image_paths=image_paths,
    output_dir='organized_dataset_random'
)


✅ Land images assigned to train: 0
✅ Land images assigned to valid: 0
✅ Land images assigned to test: 0
✅ Water images assigned to train: 0
✅ Water images assigned to valid: 0
✅ Water images assigned to test: 0
